In [ ]:
import polars as pl
import yaml
from pathlib import Path


with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

In [6]:
VARIANT = "v1"

In [ ]:
def read_stops(stop_detection_variant: str) -> pl.DataFrame:
    stops = pl.DataFrame()
    for i in range(256):
        part = pl.read_parquet(
            f"{config['dirs']['output']}/{config['city']}/stops/{stop_detection_variant}/{i:02x}.parquet"
        )
        stops = pl.concat([stops, part])
    return stops

In [ ]:
stops = read_stops(VARIANT).with_columns(
    pl.col.interval_start.dt.to_string("%Y-%m").alias("year_month")
)

In [9]:
for p in stops.partition_by("year_month"):
    path = f"{config['dirs']['output']}/{config['city']}/monthly_partitioned_stops/{VARIANT}/"
    Path(path).mkdir(exist_ok=True, parents=True)
    p_ = p.sort(["device_id", "interval_start"]).drop("year_month")
    p_.write_parquet(f"{path}/{p['year_month'].first()}.parquet")

    del p_